# Spark Exercises 01 — Spark Mindset & Basics (Fast Track)

You already know pandas and Polars, so the **syntax** here will feel familiar. This short chapter is a fast tour of the basics — but its real goal is to plant the **one idea that makes Spark different**: a DataFrame is not data sitting in your RAM, it is a *recipe* that runs on a cluster, and **nothing runs until you ask for a result** (an *action*).

**Data file:** `data/orders.csv` — 8,000 rows, one row per order line.

---

> **Solutions version** — try it yourself first from `Exercises.ipynb`.

### 1. **Setup** — open a local `SparkSession` (already written for you). In Foundry the session is created for you; locally we open one ourselves. `F` is the functions module — almost every column expression lives there.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("spark-course")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Spark 3.5.1


### 2. Read `data/orders.csv` into a DataFrame called `orders` (`header=True`, `inferSchema=True`).

In [2]:
orders = spark.read.csv("data/orders.csv", header=True, inferSchema=True)

### 3. Now just evaluate `orders`. **Surprise:** unlike pandas/Polars it does **not** print the rows — only the schema. A Spark DataFrame is a plan, not data.

In [3]:
orders

DataFrame[order_id: string, customer_id: string, order_ts: timestamp, product_sku: string, quantity: int, unit_price: double, channel: string, payment_method: string, status: string, discount_code: string]

### 4. To actually see data you call an **action**. Show the first 5 rows with `show(5)`.

In [4]:
orders.show(5)

+----------+-----------+-------------------+-----------+--------+----------+-------+--------------+---------+-------------+
|  order_id|customer_id|           order_ts|product_sku|quantity|unit_price|channel|payment_method|   status|discount_code|
+----------+-----------+-------------------+-----------+--------+----------+-------+--------------+---------+-------------+
|ORD-000001|  CUST-0092|2024-07-24 11:01:00|      BK302|      10|     32.58|    web|          card|completed|          VIP|
|ORD-000002|  CUST-0016|2024-05-09 14:53:00|      ST500|      22|      1.43|    web|      applepay|completed|         NULL|
|ORD-000003|  CUST-0289|2024-05-12 20:54:00|      PB024|       9|      1.14|    web|        paypal|completed|         NULL|
|ORD-000004|  CUST-0315|2024-07-07 10:26:00|      MG010|      21|      7.12|    app|          card|completed|         NULL|
|ORD-000005|  CUST-0356|2024-10-15 08:16:00|      PT044|      23|      8.17|  store|        paypal|completed|         NULL|
+-------

### 5. Wide rows get truncated. Show **3 rows vertically** (`show(3, vertical=True)`) — much easier to read.

In [5]:
orders.show(3, vertical=True)

-RECORD 0-----------------------------
 order_id       | ORD-000001          
 customer_id    | CUST-0092           
 order_ts       | 2024-07-24 11:01:00 
 product_sku    | BK302               
 quantity       | 10                  
 unit_price     | 32.58               
 channel        | web                 
 payment_method | card                
 status         | completed           
 discount_code  | VIP                 
-RECORD 1-----------------------------
 order_id       | ORD-000002          
 customer_id    | CUST-0016           
 order_ts       | 2024-05-09 14:53:00 
 product_sku    | ST500               
 quantity       | 22                  
 unit_price     | 1.43                
 channel        | web                 
 payment_method | applepay            
 status         | completed           
 discount_code  | NULL                
-RECORD 2-----------------------------
 order_id       | ORD-000003          
 customer_id    | CUST-0289           
 order_ts       | 2024-05

### 6. Print the **schema** (column names + types) with `printSchema()`. Notice the type Spark inferred for `order_ts`.

In [6]:
orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- product_sku: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- channel: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- status: string (nullable = true)
 |-- discount_code: string (nullable = true)



### 7. How many **rows**? Use `count()`. (This is an action — Spark scans the whole dataset to answer.)

In [7]:
orders.count()

8000

### 8. What are the **column names**, and how many columns are there?

In [8]:
print(orders.columns)
print(len(orders.columns))

['order_id', 'customer_id', 'order_ts', 'product_sku', 'quantity', 'unit_price', 'channel', 'payment_method', 'status', 'discount_code']
10


### 9. Select only `order_id`, `product_sku`, `quantity` and show 5 rows. Note there is no `df[['a','b']]` in Spark — you use `select`.

In [9]:
orders.select("order_id", "product_sku", "quantity").show(5)

+----------+-----------+--------+
|  order_id|product_sku|quantity|
+----------+-----------+--------+
|ORD-000001|      BK302|      10|
|ORD-000002|      ST500|      22|
|ORD-000003|      PB024|       9|
|ORD-000004|      MG010|      21|
|ORD-000005|      PT044|      23|
+----------+-----------+--------+
only showing top 5 rows



### 10. Get summary statistics for `quantity` and `unit_price` with `describe()`.

In [10]:
orders.describe("quantity", "unit_price").show()

+-------+-----------------+------------------+
|summary|         quantity|        unit_price|
+-------+-----------------+------------------+
|  count|             8000|              8000|
|   mean|        11.926625| 19.32074000000005|
| stddev|7.384074530972713|24.302241335299517|
|    min|               -5|              1.02|
|    max|               24|            131.83|
+-------+-----------------+------------------+



### 11. How many **distinct products** (`product_sku`) appear?

In [11]:
orders.select("product_sku").distinct().count()

20

### 12. How many orders are there **per `status`**? (`groupBy(...).count()`)

In [12]:
orders.groupBy("status").count().show()

+---------+-----+
|   status|count|
+---------+-----+
|completed| 6533|
|  pending|  762|
| returned|  306|
|cancelled|  399|
+---------+-----+



### 13. Show 5 orders whose `status` is `completed`. Use `filter` with `F.col(...)`.

In [13]:
orders.filter(F.col("status") == "completed").show(5)

+----------+-----------+-------------------+-----------+--------+----------+-------+--------------+---------+-------------+
|  order_id|customer_id|           order_ts|product_sku|quantity|unit_price|channel|payment_method|   status|discount_code|
+----------+-----------+-------------------+-----------+--------+----------+-------+--------------+---------+-------------+
|ORD-000001|  CUST-0092|2024-07-24 11:01:00|      BK302|      10|     32.58|    web|          card|completed|          VIP|
|ORD-000002|  CUST-0016|2024-05-09 14:53:00|      ST500|      22|      1.43|    web|      applepay|completed|         NULL|
|ORD-000003|  CUST-0289|2024-05-12 20:54:00|      PB024|       9|      1.14|    web|        paypal|completed|         NULL|
|ORD-000004|  CUST-0315|2024-07-07 10:26:00|      MG010|      21|      7.12|    app|          card|completed|         NULL|
|ORD-000005|  CUST-0356|2024-10-15 08:16:00|      PT044|      23|      8.17|  store|        paypal|completed|         NULL|
+-------

### 14. Add a new column `line_total = quantity * unit_price` with `withColumn`, then show `order_id`, `quantity`, `unit_price`, `line_total` for 5 rows.

In [14]:
orders.withColumn("line_total", F.col("quantity") * F.col("unit_price")) \
      .select("order_id", "quantity", "unit_price", "line_total").show(5)

+----------+--------+----------+------------------+
|  order_id|quantity|unit_price|        line_total|
+----------+--------+----------+------------------+
|ORD-000001|      10|     32.58|325.79999999999995|
|ORD-000002|      22|      1.43|31.459999999999997|
|ORD-000003|       9|      1.14|             10.26|
|ORD-000004|      21|      7.12|            149.52|
|ORD-000005|      23|      8.17|            187.91|
+----------+--------+----------+------------------+
only showing top 5 rows



### 15. Confirm Spark DataFrames are **immutable**: print `orders.columns` again — `line_total` is **not** there, because `withColumn` returned a *new* DataFrame and we never reassigned `orders`.

In [15]:
orders.columns

['order_id',
 'customer_id',
 'order_ts',
 'product_sku',
 'quantity',
 'unit_price',
 'channel',
 'payment_method',
 'status',
 'discount_code']

### 16. `show()` only prints. To pull rows back into Python as objects, use `collect()` (or `take`). Get the **first 3 rows** as a Python list of `Row` objects with `orders.limit(3).collect()`. ⚠️ Never `collect()` a huge DataFrame — it all flows to the driver's memory.

In [16]:
orders.limit(3).collect()

[Row(order_id='ORD-000001', customer_id='CUST-0092', order_ts=datetime.datetime(2024, 7, 24, 11, 1), product_sku='BK302', quantity=10, unit_price=32.58, channel='web', payment_method='card', status='completed', discount_code='VIP'),
 Row(order_id='ORD-000002', customer_id='CUST-0016', order_ts=datetime.datetime(2024, 5, 9, 14, 53), product_sku='ST500', quantity=22, unit_price=1.43, channel='web', payment_method='applepay', status='completed', discount_code=None),
 Row(order_id='ORD-000003', customer_id='CUST-0289', order_ts=datetime.datetime(2024, 5, 12, 20, 54), product_sku='PB024', quantity=9, unit_price=1.14, channel='web', payment_method='paypal', status='completed', discount_code=None)]